# 09.1 音频指纹：从五秒片段到曲库命中

本 Notebook 讨论的听歌识曲要求查询与库内条目来自同一源录音。查询可能只有几秒钟，并经过环境播放、麦克风采集、重采样或有损压缩，但其内容仍是库内录音的一段。

这些处理不会改变来源录音的身份，指纹算法却只能在一定扰动范围内保持识别。若换一位歌手重新演唱同一作品，作品关系仍在，但已不是同一份源录音，需要使用翻唱识别方法。

任务包含两个问题：查询是否来自曲库中的某份录音；若是，它在原录音中从何时开始。朴素的波形逐点比较既耗时，也对时间偏移和信号扰动敏感。

这里把频谱图压缩成稀疏的星座图（constellation map），只记录较显著的局部峰。峰值配对生成离散哈希键；查询命中曲库记录后，按曲目和时间偏移投票。同源片段的命中通常集中在相近偏移处，这是地标哈希（landmark hashing）的核心。


## 0. 环境与数据

曲库取 MUSDB18-HQ 的十首完整曲目，另备一条库外音频作负例。MUSDB18-HQ 各曲目的来源和许可状态并不完全相同，官方数据页也要求按学术用途申请访问。下面的清单按官方逐曲信息标注，不宜用其中一种许可概括整套数据。

首个代码单元检查依赖、外部命令和数据文件。缺少项目依赖时，应按检查结果补齐后重新运行。


In [ ]:
from pathlib import Path
import sys
import time

# 路径推断：从 cwd 向上找含 CODE/chapter09/_common 的目录；ROOT 指向 CODE/chapter09/
_p = Path.cwd()
while not (_p / "CODE" / "chapter09" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter09/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter09"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from _common.audio_io import load_audio  # audio_io 先设 NUMBA_CACHE_DIR，须早于 librosa 导入
from _common.env_check import check_notebook_env
from _common.paths import portable_path
from _common.plotting import BAR_GRAY, GRAY_IMAGE_CMAP, LINE_GRAYS, finish_figure, setup_plot_style

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, signal

check_notebook_env("09_1_audio_fingerprint")

SR = 22050            # 全章统一采样率
N_FFT = 2048          # 频率分辨率约 10.8 Hz/bin
HOP = 512             # 帧率约 43.1 Hz
FREQ_RES = SR / N_FFT
FRAME_RATE = SR / HOP
PEAKS_PER_FRAME = 5   # 密度参数：每个时间片保留的峰值数
FAN_OUT = 5           # 每个锚点最多配对的目标峰数
MIN_DT = 1            # 目标峰最早在锚点之后 1 帧
MAX_DT = 45           # 目标区长约 1 秒
MATCH_MIN_VOTES = 20  # 判"无匹配"的票高下限，依据见负例对照单元
QUERY_SEC = 5.0

DATASETS = ROOT.parent / "datasets"
MANIFEST = ROOT / "data_manifests" / "fingerprint_library.csv"
OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_AUDIO = ROOT / "output_audio" / "09_1"
for path in [OUTPUT_FIGURES, OUTPUT_TABLES, OUTPUT_AUDIO]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

library_df = pd.read_csv(MANIFEST)
library_df["exists"] = library_df["path"].map(lambda p: (ROOT.parent / p).exists())
print(f"帧率 {FRAME_RATE:.1f} Hz,频率分辨率 {FREQ_RES:.1f} Hz/bin")
library_df[["track", "role", "duration_sec", "license_status", "exists"]]


## 1. 星座图与峰值挑选

频谱图中的大量时频点没有必要直接写入指纹。局部能量峰比周围点更容易在中等噪声和编码失真下重复出现；有明确音高的乐音常在较强谐波附近产生峰，打击乐和噪声性声音也可能在瞬态或宽带结构中产生可重复峰。

峰值挑选分两步。先用最大滤波寻找时频邻域内的局部极大点，并丢弃比全局最强峰低 45 dB 以上的点；再在每个时间片内按幅度保留最强的若干个，以参数控制峰值密度。

峰值较多时，扰动后仍可配对的记录通常也较多，但哈希表更大，随机假命中也可能增加；峰值较少时则相反。Dejavu 等开源实现同样通过振幅门限和配对扇出调节这类权衡。

下图把保留的峰叠加在 25 秒频谱图上。后续配对与哈希只需要这些峰的时频位置，不再使用完整的稠密频谱图。


In [ ]:
def stft_db(y):
    # 幅度谱转 dB，每首歌以自身峰值为 0 dB 参考
    spec = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP, window="hann"))
    return librosa.amplitude_to_db(spec, ref=np.max)


def find_peaks(db, top_n, threshold_db=-45.0, neighborhood=21):
    # 矩形邻域的二维最大滤波可拆成两次一维滤波，结果相同但快得多
    max_f = ndimage.maximum_filter1d(db, size=neighborhood, axis=0, mode="nearest")
    max_tf = ndimage.maximum_filter1d(max_f, size=neighborhood, axis=1, mode="nearest")
    is_peak = (db == max_tf) & (db >= db.max() + threshold_db)
    peaks = []
    for t in range(db.shape[1]):
        cols = np.flatnonzero(is_peak[:, t])
        if cols.size > top_n:
            cols = cols[np.argsort(db[cols, t])[-top_n:]]
        peaks.extend((t, int(f), float(db[f, t])) for f in cols)
    return sorted(peaks)


excerpt_path = ROOT.parent / library_df.loc[0, "path"]
y_excerpt, _ = load_audio(excerpt_path, sr=SR, offset=10.0, duration=25.0)
db_excerpt = stft_db(y_excerpt)

rows = []
for n in (2, 3, 5, 8):
    peaks_n = find_peaks(db_excerpt, top_n=n)
    rows.append({"top_n": n, "peaks_total": len(peaks_n), "peaks_per_sec": round(len(peaks_n) / 25.0, 1)})
print(pd.DataFrame(rows).to_string(index=False))

peaks_excerpt = find_peaks(db_excerpt, top_n=PEAKS_PER_FRAME)
extent = [0, db_excerpt.shape[1] / FRAME_RATE, 0, SR / 2000]  # 频率轴画到 11 kHz，单位 kHz
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.imshow(db_excerpt, origin="lower", aspect="auto", extent=extent, cmap=GRAY_IMAGE_CMAP, vmin=-70, vmax=0)
ax.scatter(
    [p[0] / FRAME_RATE for p in peaks_excerpt],
    [p[1] * FREQ_RES / 1000 for p in peaks_excerpt],
    s=4, c="0.05", linewidths=0,
)
ax.set_xlabel("时间（秒）")
ax.set_ylabel("频率（kHz）")
finish_figure(fig, OUTPUT_FIGURES / "09_1_constellation.png")
plt.show()


## 2. 配对、哈希与离散键

单个峰的辨识力有限。五秒查询中即使有一个峰与库内频点重合，也不足以确认来源。地标哈希把峰两两配对：取一个峰作锚点，在其后约一秒的目标区内至多选择若干个目标峰。

每个（锚点，目标）对生成一条记录。键由锚点频率、目标频率和帧差组成；锚点在曲内的绝对帧号另存为值，供投票时计算查询与曲库之间的时间偏移。

下图放大一段星座图。圆圈标出锚点，虚线框表示目标区，连线指向实际配到的目标峰。


In [ ]:
def refine_freq_hz(db, t, f):
    # 抛物线插值的亚 bin 频率(Hz)；边缘 bin 退化为 bin 中心
    if f <= 0 or f >= db.shape[0] - 1:
        return f * FREQ_RES
    a, b, c = db[f - 1, t], db[f, t], db[f + 1, t]
    denom = a - 2 * b + c
    if denom >= 0:
        return f * FREQ_RES
    return (f + 0.5 * (a - c) / denom) * FREQ_RES


def pair_hashes(peaks, fan_out, db=None, refine=False):
    # 每个锚点向后排对：同一帧内峰按频率升序，帧按时间升序，窗口外即停
    hashes = []
    for i, (t1, f1, _) in enumerate(peaks):
        taken = 0
        for t2, f2, _ in peaks[i + 1:]:
            dt = t2 - t1
            if dt > MAX_DT:
                break
            if dt < MIN_DT:
                continue
            if refine:
                key = (refine_freq_hz(db, t1, f1), refine_freq_hz(db, t2, f2), dt / FRAME_RATE)
            else:
                key = (f1, f2, dt)
            hashes.append((key, t1))
            taken += 1
            if taken >= fan_out:
                break
    return hashes


# 放大 4 秒，标出一个锚点、目标区与实际配到的目标峰
# 图示锚点要求：目标峰全部落在可视频带内，且在时间轴上尽量散开
f_hi_khz = 6.0
anchor, partners, best_spread = None, [], -1
for i, (t1, f1, _) in enumerate(peaks_excerpt):
    t1s = t1 / FRAME_RATE
    if not (2.0 < t1s < 22.0 and 800 < f1 * FREQ_RES < 3000):
        continue
    cand = [p for p in peaks_excerpt[i + 1:] if MIN_DT <= p[0] - t1 <= MAX_DT][:FAN_OUT]
    if len(cand) < 4 or max(p[1] for p in cand) * FREQ_RES > 5200:
        continue
    spread = max(p[0] for p in cand) - t1
    if spread > best_spread:
        anchor, partners, best_spread = (t1, f1), cand, spread
if anchor is None:  # 兜底：片段中部第一个峰
    i0 = min(range(len(peaks_excerpt)), key=lambda i: abs(peaks_excerpt[i][0] / FRAME_RATE - 12.5))
    t1, f1, _ = peaks_excerpt[i0]
    anchor = (t1, f1)
    partners = [p for p in peaks_excerpt[i0 + 1:] if MIN_DT <= p[0] - t1 <= MAX_DT][:FAN_OUT]
t_center = anchor[0] / FRAME_RATE
t_lo, t_hi = t_center - 0.6, t_center + 3.4

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.imshow(db_excerpt, origin="lower", aspect="auto", extent=extent, cmap=GRAY_IMAGE_CMAP, vmin=-70, vmax=0)
ax.set_xlim(t_lo, t_hi)
ax.set_ylim(0, f_hi_khz)
window_peaks = [p for p in peaks_excerpt if t_lo <= p[0] / FRAME_RATE <= t_hi and p[1] * FREQ_RES <= f_hi_khz * 1000]
ax.scatter(
    [p[0] / FRAME_RATE for p in window_peaks],
    [p[1] * FREQ_RES / 1000 for p in window_peaks],
    s=10, c="0.05", linewidths=0,
)
ax.add_patch(
    plt.Rectangle(
        ((anchor[0] + MIN_DT) / FRAME_RATE, 0),
        (MAX_DT - MIN_DT) / FRAME_RATE,
        f_hi_khz,
        fill=False,
        edgecolor="0.05",
        linestyle="--",
        linewidth=1.0,
    )
)
for t2, f2, _ in partners:
    ax.plot(
        [anchor[0] / FRAME_RATE, t2 / FRAME_RATE],
        [anchor[1] * FREQ_RES / 1000, f2 * FREQ_RES / 1000],
        color="0.10",
        linewidth=1.2,
        zorder=4,
    )
ax.scatter(
    [anchor[0] / FRAME_RATE],
    [anchor[1] * FREQ_RES / 1000],
    s=90, facecolors="none", edgecolors="0.05", linewidths=1.5, zorder=5,
)
ax.set_xlabel("时间（秒）")
ax.set_ylabel("频率（kHz）")
finish_figure(fig, OUTPUT_FIGURES / "09_1_anchor_pairs.png")
plt.show()


精确哈希查询要求建库端与查询端生成相同的离散键。本实现直接使用频谱 bin 的整数序号和整数帧差。字典键并非必须是整数，但未经量化的连续估计不适合直接作精确键。

对照条件把峰值频率换算成 Hz，并用抛物线插值细化到亚 bin 精度。查询起点通常不落在建库端短时傅里叶变换的同一帧网格上，同一谱峰两次插值得到的数值可能略有不同，精确查询便会漏掉本应匹配的记录。

改进方法可以把插值结果重新量化，或查询相邻键，而不是放弃精细估计。下面只比较“整数离散键”与“未经量化的浮点键”。结果只表明后一种键不适合当前的精确查询，不能否定浮点估计本身。


In [ ]:
def build_track_db(y, refine=False):
    db = stft_db(y)
    peaks = find_peaks(db, top_n=PEAKS_PER_FRAME)
    table = {}
    for key, t1 in pair_hashes(peaks, FAN_OUT, db=db, refine=refine):
        table.setdefault(key, []).append(t1)
    return table


def count_hits(table, y, refine=False):
    db = stft_db(y)
    peaks = find_peaks(db, top_n=PEAKS_PER_FRAME)
    return sum(len(table.get(key, ())) for key, _ in pair_hashes(peaks, FAN_OUT, db=db, refine=refine))


# 同一首歌建两种键的库，用同一条查询对照
demo_path = ROOT.parent / library_df.loc[1, "path"]
y_demo, _ = load_audio(demo_path, sr=SR)
db_int = build_track_db(y_demo, refine=False)
db_float = build_track_db(y_demo, refine=True)

query_start = 33.77  # 故意不落在 hop 网格上
y_clean, _ = load_audio(demo_path, sr=SR, offset=query_start, duration=QUERY_SEC)
rng = np.random.default_rng(0)
snr_db = 10.0
noise_scale = np.sqrt(np.mean(y_clean**2) / 10 ** (snr_db / 10))
y_noisy = y_clean + noise_scale * rng.standard_normal(y_clean.size).astype(np.float32)

rows = [
    {"键类型": "整数 bin 键", "查询": "干净片段", "命中哈希数": count_hits(db_int, y_clean)},
    {"键类型": "整数 bin 键", "查询": f"加噪 {snr_db:.0f} dB", "命中哈希数": count_hits(db_int, y_noisy)},
    {"键类型": "浮点插值键", "查询": "干净片段", "命中哈希数": count_hits(db_float, y_clean)},
    {"键类型": "浮点插值键", "查询": f"加噪 {snr_db:.0f} dB", "命中哈希数": count_hits(db_float, y_noisy)},
]
print(pd.DataFrame(rows).to_string(index=False))


在当前对照中，整数键对干净查询命中 387 条哈希，对 10 dB 加噪查询命中 146 条；未经量化的浮点键在两种条件下都没有命中。差别来自键的可重复性，而不是插值精度没有价值。

Wang 的论文说明，可把两个频率分量和时间差打包进一个 32 位无符号整数，并以每个分量约 10 位说明信息量；论文没有规定固定的 `10+10+12` 分配。时间偏移与曲目编号还可另存于一个 32 位字。本 Notebook 用 Python 元组保留三个可检查的分量。

接下来对十首曲目逐首提峰、配对并建表。字典的键是（锚点 bin，目标 bin，帧差），值是（曲目，锚点帧号）的列表。同一键出现在不同曲目或位置时，相应记录会存入同一个列表。


In [ ]:
hash_db = {}
build_rows = []
t_start = time.perf_counter()
for _, row in library_df[library_df["role"] == "library"].iterrows():
    track_id = row["track"]
    y_track, _ = load_audio(ROOT.parent / row["path"], sr=SR)
    db_track = stft_db(y_track)
    peaks_track = find_peaks(db_track, top_n=PEAKS_PER_FRAME)
    n_hashes = 0
    for key, t1 in pair_hashes(peaks_track, FAN_OUT):
        hash_db.setdefault(key, []).append((track_id, t1))
        n_hashes += 1
    build_rows.append(
        {"track": track_id, "时长(秒)": round(row["duration_sec"]), "峰值数": len(peaks_track), "哈希条数": n_hashes}
    )
elapsed = time.perf_counter() - t_start
n_entries = sum(len(v) for v in hash_db.values())
print(pd.DataFrame(build_rows).to_string(index=False))
print(f"建库耗时 {elapsed:.1f} 秒;唯一键 {len(hash_db):,} 个,键值对共 {n_entries:,} 条")


## 3. 投票与时间偏移直方图

查询采用与建库相同的提峰、配对和哈希流程。每条命中返回（曲目，库端锚点帧号）；减去查询端锚点帧号，得到一个时间偏移。若查询截自库内录音，正确命中通常给出相近偏移。

每首歌分别累积偏移直方图。正确候选通常在片段真实起点附近形成主峰：主峰所在曲目给出身份，横坐标给出起始位置，峰高是该偏移的票数。其他曲目也可能偶然命中，但偏移通常更分散。

当前库内示例的主峰为 76 票，库外负例的最高票为 5。代码暂以 20 票演示拒识。实际阈值应使用多条、具有代表性的库内与库外验证查询标定。这里只有一条负例，数据不足以估计漏检率或误报率，也不应把 20 票沿用到其他曲库。


In [ ]:
def query_fingerprint(hash_db, y_query):
    db_q = stft_db(y_query)
    peaks_q = find_peaks(db_q, top_n=PEAKS_PER_FRAME)
    votes = {}
    for key, t1q in pair_hashes(peaks_q, FAN_OUT):
        for track_id, t1lib in hash_db.get(key, ()):
            votes.setdefault(track_id, []).append(t1lib - t1q)
    scored = []
    for track_id, offsets in votes.items():
        offsets = np.asarray(offsets)
        hist, edges = np.histogram(offsets, bins=np.arange(offsets.min(), offsets.max() + 2))
        peak_bin = int(hist.argmax())
        scored.append(
            {
                "track": track_id,
                "peak_votes": int(hist[peak_bin]),
                "offset_sec": round(float((edges[peak_bin] + edges[peak_bin + 1]) / 2 / FRAME_RATE), 2),
                "total_votes": int(offsets.size),
                "offsets": offsets,
            }
        )
    scored.sort(key=lambda r: r["peak_votes"], reverse=True)
    return scored


def report_query(label, y_query, truth=None):
    scored = query_fingerprint(hash_db, y_query)
    best = scored[0] if scored else None
    matched = best is not None and best["peak_votes"] >= MATCH_MIN_VOTES
    verdict = best["track"] if matched else "无匹配"
    print(f"{label}：判定 {verdict}", end="")
    if best is not None:
        print(f"；最高票 {best['peak_votes']}，偏移 {best['offset_sec']} 秒，总票 {best['total_votes']}")
    else:
        print("；库内无任何命中")
    if truth is not None:
        print(f"  真实答案：{truth}（{'正确' if verdict == truth else '错误'}）")
    return scored, best


# 正例：库内截取 5 秒，起点不在 hop 网格上
hit_row = library_df.loc[2]
y_hit, _ = load_audio(ROOT.parent / hit_row["path"], sr=SR, offset=47.31, duration=QUERY_SEC)
scored_hit, best_hit = report_query("库内片段", y_hit, truth=hit_row["track"])

# 负例：库外音频
neg_row = library_df[library_df["role"] == "negative_query"].iloc[0]
y_neg, _ = load_audio(ROOT.parent / neg_row["path"], sr=SR, offset=60.13, duration=QUERY_SEC)
scored_neg, best_neg = report_query("库外片段", y_neg)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, scored, title in [
    (axes[0], scored_hit, f"库内片段（真实 {hit_row['track']}）"),
    (axes[1], scored_neg, "库外片段"),
]:
    if scored:
        top = scored[0]
        ax.hist(
            top["offsets"] / FRAME_RATE,
            bins=np.arange(top["offsets"].min(), top["offsets"].max() + 2) / FRAME_RATE,
            color=BAR_GRAY,
            edgecolor="0.15",
            linewidth=0.3,
        )
        if top["peak_votes"] >= MATCH_MIN_VOTES:
            ax.set_title(f"{title}，判定 {top['track'][:20]}", fontsize=9)
        else:
            ax.set_title(f"{title}，最高票仅 {top['peak_votes']} 票，判无匹配", fontsize=9)
    else:
        ax.set_title(f"{title}，库内无任何命中", fontsize=9)
    ax.set_xlabel("时间偏移（秒）")
axes[0].set_ylabel("票数")
finish_figure(fig, OUTPUT_FIGURES / "09_1_offset_histograms.png")
plt.show()


## 4. 扰动下的鲁棒性

本节选择加性噪声、窄带滤波和低码率压缩作定量示例。噪声实验逐级降低信噪比，另外两项各取一个设置：带通滤波保留 300–3400 Hz，近似常见窄带电话语音频段；MP3 以 32 kbps 编码后再解码。

扫描只含六条库内查询，识别率用于展示当前实现随扰动增强而失效的过程，不代表一般场景中的准确率。窄带和低码率实验也只检验一个参数点。

变速与变调会使峰在时间轴或频率轴上系统性移动。它们仍可能来自同一源录音，但当前峰值配对键没有为这些变换设计不变性。Panako 等方法使用对数频率轴上的峰组三元关系，使对数频率差对整体频移更稳定、时间差比值对均匀时间伸缩更稳定。可容忍范围仍取决于扰动类型、查询长度和实验条件。


In [ ]:
import subprocess
import tempfile

import soundfile as sf


def add_noise(y, snr_db, rng):
    scale = np.sqrt(np.mean(y**2) / 10 ** (snr_db / 10))
    return y + scale * rng.standard_normal(y.size).astype(np.float32)


def bandpass(y, lo=300.0, hi=3400.0):
    sos = signal.butter(4, [lo, hi], btype="band", fs=SR, output="sos")
    return signal.sosfiltfilt(sos, y).astype(np.float32)


def mp3_roundtrip(y, bitrate="32k"):
    with tempfile.TemporaryDirectory() as tmp:
        src, enc, dec = Path(tmp) / "q.wav", Path(tmp) / "q.mp3", Path(tmp) / "q_dec.wav"
        sf.write(src, y, SR)
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-codec:a", "libmp3lame", "-b:a", bitrate, str(enc)],
            check=True,
        )
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(enc), str(dec)], check=True)
        out, _ = load_audio(dec, sr=SR)
    return out[: y.size]


def evaluate_queries(perturbed):
    hits, votes = 0, []
    for truth, y_q in perturbed:
        scored = query_fingerprint(hash_db, y_q)
        best = scored[0] if scored else None
        votes.append(best["peak_votes"] if best else 0)
        if best is not None and best["peak_votes"] >= MATCH_MIN_VOTES and best["track"] == truth:
            hits += 1
    return hits / len(perturbed), float(np.mean(votes))


# 六首库内曲目各截一条 5 秒查询，起点不刻意对齐
sweep_queries = []
for _, row in library_df[library_df["role"] == "library"].head(6).iterrows():
    offset = round(0.42 * row["duration_sec"], 2)
    y_q, _ = load_audio(ROOT.parent / row["path"], sr=SR, offset=offset, duration=QUERY_SEC)
    sweep_queries.append((row["track"], y_q))

rng = np.random.default_rng(20240901)
robust_rows = []
for snr in (20, 15, 10, 5, 0, -5):
    perturbed = [(truth, add_noise(y_q, snr, rng)) for truth, y_q in sweep_queries]
    rate, mean_votes = evaluate_queries(perturbed)
    robust_rows.append({"SNR(dB)": snr, "识别率": round(rate, 3), "平均最高票": round(mean_votes, 1)})
robust_df = pd.DataFrame(robust_rows)
print(robust_df.to_string(index=False))

fixed_rows = []
for label, fn in [("干净基线", lambda y: y), ("带通 300–3400 Hz", bandpass), ("MP3 32 kbps", mp3_roundtrip)]:
    rate, mean_votes = evaluate_queries([(truth, fn(y_q)) for truth, y_q in sweep_queries])
    fixed_rows.append({"扰动": label, "识别率": round(rate, 3), "平均最高票": round(mean_votes, 1)})
fixed_df = pd.DataFrame(fixed_rows)
print(fixed_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(robust_df["SNR(dB)"], robust_df["识别率"], marker="o", color=LINE_GRAYS[0])
axes[0].set_xlabel("信噪比（dB）")
axes[0].set_ylabel("识别率")
axes[0].set_ylim(-0.05, 1.05)
axes[1].plot(robust_df["SNR(dB)"], robust_df["平均最高票"], marker="s", color=LINE_GRAYS[0])
axes[1].axhline(MATCH_MIN_VOTES, color=LINE_GRAYS[3], linestyle="--", linewidth=1, label=f"无匹配下限 {MATCH_MIN_VOTES} 票")
axes[1].set_xlabel("信噪比（dB）")
axes[1].set_ylabel("平均最高票")
axes[1].legend(fontsize=8)
finish_figure(fig, OUTPUT_FIGURES / "09_1_robustness_snr.png")
plt.show()


## 5. 同一首歌的不同版本

前面的识别都建立在查询与库内条目来自同一源录音上。这里用受控的不同版本检查边界：两版《茉莉花》保留作品关系，但同时改变音色、整体音高和时间尺度。

第 3 章的渲染工具链生成长笛原版和小提琴改编版。改编版移高四个半音、加速到 1.37 倍，并加入轻微节奏弹性。原版片段在新增曲库条目上得到 200 票，改编版只有 1 票，因而判为无匹配。

原版直方图除主峰外还有较低次峰。旋律中的重复句式可能使相似局部结构在多个偏移上匹配，但仅凭直方图不能排除其他重复时频结构或随机碰撞。当前结果说明这套峰值配对指纹比较的是局部声学峰及其时频关系。

翻唱识别需要更抽象的旋律、和声或结构表示，并处理调性与时间差异。后续 Notebook 将沿这一方向展开。


In [ ]:
import warnings

from _common.cover_render import CoverSpec, render_cover

# 民歌 MIDI 的 tempo 事件不在第 0 轨，pretty_midi 会告警，渲染不受影响
warnings.filterwarnings("ignore", message=r"Tempo, Key or Time signature.*", category=RuntimeWarning)

molihua_midi = DATASETS / "melodies" / "茉莉花.midi"
original_wav = OUTPUT_AUDIO / "molihua_original.wav"
cover_wav = OUTPUT_AUDIO / "molihua_cover.wav"
render_cover(molihua_midi, CoverSpec(version="v0_original", seed=7), original_wav, sr=SR)
render_cover(
    molihua_midi,
    CoverSpec(version="mini_cover", program=40, transpose=4, tempo_ratio=1.37, rubato=0.03, seed=7),
    cover_wav,
    sr=SR,
)
y_orig, _ = load_audio(original_wav, sr=SR)
y_cover, _ = load_audio(cover_wav, sr=SR)

# 原版入曲库，作为第十一条条目
db_o = stft_db(y_orig)
peaks_o = find_peaks(db_o, top_n=PEAKS_PER_FRAME)
for key, t1 in pair_hashes(peaks_o, FAN_OUT):
    hash_db.setdefault(key, []).append(("茉莉花(合成原版)", t1))

def boundary_query(label, y_full):
    offset = round(0.3 * len(y_full) / SR, 2)
    y_q = y_full[int(offset * SR) : int((offset + QUERY_SEC) * SR)]
    scored = query_fingerprint(hash_db, y_q)
    best = scored[0] if scored else None
    verdict = best["track"] if best is not None and best["peak_votes"] >= MATCH_MIN_VOTES else "无匹配"
    molihua = next((r for r in scored if r["track"] == "茉莉花(合成原版)"), None)
    print(f"{label}：判定 {verdict}；投给茉莉花条目的最高票 {molihua['peak_votes'] if molihua else 0}")
    return scored, verdict

scored_bo, verdict_bo = boundary_query("原版片段", y_orig)
scored_bc, verdict_bc = boundary_query("改编版片段", y_cover)

boundary_rows = [
    {"查询": "原版片段", "判定": verdict_bo, "茉莉花条目最高票": next((r["peak_votes"] for r in scored_bo if r["track"] == "茉莉花(合成原版)"), 0)},
    {
        "查询": "改编版片段",
        "判定": verdict_bc,
        "茉莉花条目最高票": next((r["peak_votes"] for r in scored_bc if r["track"] == "茉莉花(合成原版)"), 0),
    },
]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, scored, title in [(axes[0], scored_bo, "原版片段查询"), (axes[1], scored_bc, "改编版片段查询")]:
    target = next((r for r in scored if r["track"] == "茉莉花(合成原版)"), scored[0] if scored else None)
    if target is not None:
        ax.hist(
            target["offsets"] / FRAME_RATE,
            bins=np.arange(target["offsets"].min(), target["offsets"].max() + 2) / FRAME_RATE,
            color=BAR_GRAY,
            edgecolor="0.15",
            linewidth=0.3,
        )
        ax.set_title(f"{title}，茉莉花条目最高票 {target['peak_votes']}", fontsize=9)
    else:
        ax.set_title(f"{title}，茉莉花条目零命中", fontsize=9)
    ax.set_xlabel("时间偏移（秒）")
axes[0].set_ylabel("票数")
finish_figure(fig, OUTPUT_FIGURES / "09_1_cover_boundary.png")
plt.show()


## 6. 工程现实

Wang 在 2003 年 ISMIR 论文中描述了 Shazam 当时使用的地标哈希算法。Shazam 于 2002 年在英国推出拨号识曲服务：用户拨打 2580，让电话采集音乐，随后收到曲名短信。

工业实现还要处理存储、延迟和候选验证。曲库扩大后，哈希表可能需要压缩、分片或分布式存储；投票得到的候选也可再检查时间一致性。Dejavu 与 audfprint 采用相近的基本流程，而本 Notebook 只处理单机内存中的小型曲库。

音频指纹也可以通过训练获得。Google Now Playing 使用神经网络在设备上生成指纹，并与本地曲库匹配；神经音频指纹通常把短片段映射为低维嵌入，可在训练中用噪声、房间响应等增广塑造鲁棒性。具体不变量仍受网络结构、训练目标和数据共同影响。

部署时需按曲库和场景调整峰值密度、配对扇出、键的存储方式与拒识阈值。更换曲库、查询长度或峰值参数后，应使用独立验证集重新评估。


In [ ]:
pd.DataFrame(build_rows).to_csv(OUTPUT_TABLES / "09_1_library_stats.csv", index=False)
robust_df.to_csv(OUTPUT_TABLES / "09_1_robustness_snr.csv", index=False)
fixed_df.to_csv(OUTPUT_TABLES / "09_1_fixed_perturbations.csv", index=False)
pd.DataFrame(boundary_rows).to_csv(OUTPUT_TABLES / "09_1_cover_boundary.csv", index=False)
print("表格已写入", rel(OUTPUT_TABLES))
print("图已写入", rel(OUTPUT_FIGURES))
